In [4]:
import pandas as pd
import random

metrics = ["revenue", "recharge amount", "data volume", "og call revenue", "incoming call MOU", "outgoing call MOU", "data usage", "account balance"]
events = ["purchases", "action keys", "bonuses", "promotions", "billing events", "campaign responses"]
time_units = ["days", "months", "weeks"]
regions = ["North", "South", "Region X", "Northoman"]
products = ["product id 500", "prepaid best plan", "HBB add-on", "MD03", "M138"]
attributes = ["network age", "tariff plan", "audience segment", "VALUE_SEGMENT_OVERALL"]
entities = ["subscriber", "device", "product", "MSISDN"]

def get_t1():
    templates = [
        "total {metric} in the last {n} {time_unit}",
        "average {metric} over last {n} {time_unit}",
        "count of {event} in the last {n} {time_unit}",
        "{metric} last month",
        "average weekly {metric}",
        "combined {metric} and {metric2} over last {n} {time_unit}",
        "customers whose {event} count in the last {n} {time_unit} >= {val}",
        "total {metric} for {product}",
        "sum of {metric} and {metric2} in the last 30 days"
    ]
    return random.choice(templates).format(
        metric=random.choice(metrics),
        metric2=random.choice(metrics),
        n=random.randint(2, 90),
        time_unit=random.choice(time_units),
        event=random.choice(events),
        val=random.randint(1, 100),
        product=random.choice(products)
    )

def get_t2():
    templates = [
        "customer {attribute} > {val}",
        "customer is not subscribed to {product} in the last {val} days",
        "count of {event} sent in the last {val} days",
        "{attribute} = {val}",
        "no active {product}",
        "count of {event} per {entity}",
        "count of {event} grouped by {entity}",
        "subscribed at most {val} times in the last 30 days",
        "next best offer exists",
        "{attribute} is available",
        "not subscribed to product in last 40 days"
    ]
    return random.choice(templates).format(
        attribute=random.choice(attributes),
        val=random.randint(1, 100),
        product=random.choice(products),
        event=random.choice(events),
        entity=random.choice(entities)
    )

def get_t3():
    templates = [
        "customers detected in {region} at least once in the last {n} days",
        "customers whose current location is {region}",
        "customers who have a valid ID and MAX(ID) <> NULL",
        "customers who activated {product} yesterday",
        "customers whose {metric} on event date >= {val}",
        "customers detected in {region} at least once in the last 2 months",
        "was zero or null on the most recent date in the last {n} days"
    ]
    return random.choice(templates).format(
        region=random.choice(regions),
        n=random.randint(2, 90),
        product=random.choice(products),
        metric=random.choice(metrics),
        val=random.randint(1, 100)
    )

def get_t4():
    templates = [
        "{metric} drop of more than {n}% compared to last month",
        "{metric} ratio between this month and last month",
        "percentage increase in {metric} compared to {n} months ago",
        "difference in {metric} and {metric2} from yesterday",
        "{metric} growth over the last {n} {time_unit}"
    ]
    return random.choice(templates).format(
        metric=random.choice(metrics),
        metric2=random.choice(metrics),
        n=random.randint(2, 90),
        time_unit=random.choice(time_units)
    )

def get_t5():
    templates = [
        "subscribed to {product} in last X days",
        "customers who received a bonus for {event} in the last N days",
        "purchased specified product in the last {n} months",
        "average {metric} over last NoOfDays days",
        "{metric} greater than given value",
        "Promo sent for {event} in last X days",
        "Last X days {metric}"
    ]
    return random.choice(templates).format(
        product=random.choice(products),
        event=random.choice(events),
        n=random.randint(2, 10),
        metric=random.choice(metrics)
    )

# Generate dataset
data = []
for _ in range(85):
    data.append({"input": get_t1(), "output": 1})
    data.append({"input": get_t2(), "output": 2})
    data.append({"input": get_t3(), "output": 3})
    data.append({"input": get_t4(), "output": 4})
    data.append({"input": get_t5(), "output": 5})

df = pd.DataFrame(data)

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save file
file_path = "/content/classification_dataset.csv"
df.to_csv(file_path, index=False)

print(f"Generated dataset with {len(df)} rows.")
print(f"Saved at: {file_path}")

# Optional: Download in Colab
# from google.colab import files
# files.download(file_path)

Generated dataset with 425 rows.
Saved at: /content/classification_dataset.csv


In [5]:
pip install pandas scikit-learn transformers datasets torch

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch

In [22]:
df = pd.read_csv("classification_dataset.csv")

df = df.rename(columns={"input": "text", "output": "label"})

# ✅ FIX HERE
df["label"] = df["label"] - 1

In [23]:
df = df.rename(columns={"input": "text", "output": "label"})
print(df.head())

                                                text  label
0  was zero or null on the most recent date in th...      2
1  combined data usage and outgoing call MOU over...      0
2          not subscribed to product in last 40 days      1
3  combined account balance and og call revenue o...      0
4       customers who activated HBB add-on yesterday      2


In [24]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [25]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [26]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [27]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [28]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/340 [00:00<?, ? examples/s]

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

In [29]:
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

In [30]:
train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [31]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [34]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,1.122353
2,No log,0.624440
3,No log,0.442492


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=129, training_loss=0.9414164816686349, metrics={'train_runtime': 1728.5758, 'train_samples_per_second': 0.59, 'train_steps_per_second': 0.075, 'total_flos': 67095126328320.0, 'train_loss': 0.9414164816686349, 'epoch': 3.0})

In [35]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.4424918293952942,
 'eval_runtime': 35.4573,
 'eval_samples_per_second': 2.397,
 'eval_steps_per_second': 0.31,
 'epoch': 3.0}

In [36]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    predicted_class_id = torch.argmax(outputs.logits, dim=1).item()
    return predicted_class_id + 1   # convert back to 1–5

In [39]:
print(predict("total revenue 5 months ago"))

1


In [40]:
trainer.save_model("my_bert_model")
tokenizer.save_pretrained("my_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_bert_model/tokenizer_config.json', 'my_bert_model/tokenizer.json')